In [ ]:
### Gene-Specific Variant Extraction from VCF

# This notebook filters annotated variants in a VCF file to retain only those
# that map to genes of interest. It uses PySAM for parsing and Pandas for processing.

import pysam
import re
import pandas as pd
from pathlib import Path

def extract_gene_variants(
    vcf_path,
    output_path="pm_gene_of_interest.tsv",
    gene_list=None
):
    """
    Extracts variants from a VCF that are associated with a list of genes of interest.

    Args:
        vcf_path (str or Path): Path to the annotated VCF file.
        output_path (str or Path): Output path for the filtered variant TSV.
        gene_list (list of str): List of gene names or patterns to search for.

    Returns:
        pd.DataFrame: Filtered DataFrame containing gene-specific variants.
    """

    if gene_list is None:
        gene_list = [
            "crt", "mdr", "dhfr", "dhps", "kelch13",
            "PmUG01_01020700", "PmUG01_10021600", "PmUG01_05034700",
            "PmUG01_14045500", "PmUG01_12021200"
        ]

    gene_pattern = re.compile("|".join(gene_list), re.IGNORECASE)
    vcf = pysam.VariantFile(str(vcf_path))
    seen_variants = set()
    filtered_data = []

    for record in vcf:
        chrom = record.chrom
        pos = record.pos
        ref = record.ref
        alt_alleles = record.alts
        var_type = record.info.get("TYPE", ["Unknown"])[0]
        ann_info = record.info.get("ANN")

        if not ann_info:
            continue

        for annotation in ann_info:
            fields = annotation.split("|")
            if len(fields) < 5:
                continue  # Skip malformed entries

            effect_type = fields[1]
            gene_name = fields[3]

            if gene_pattern.search(gene_name):
                for sample in record.samples:
                    genotype = record.samples[sample]["GT"]
                    if any(gt > 0 for gt in genotype if gt is not None):
                        for alt in alt_alleles:
                            variant_key = (chrom, pos, ref, alt, sample)
                            if variant_key not in seen_variants:
                                seen_variants.add(variant_key)
                                filtered_data.append([
                                    chrom, pos, ref, alt, var_type,
                                    gene_name, effect_type, sample
                                ])

    df = pd.DataFrame(filtered_data, columns=[
        "Chromosome", "Position", "Ref", "Alt", "Variant_Type",
        "Gene", "Effect", "Sample"
    ])

    # Clean gene fields
    df["Gene"] = df["Gene"].str.replace(r"^exon-", "", regex=True)
    df["Gene"] = df["Gene"].str.replace(r"\\.\d+-\d+$", "", regex=True)

    # Save to file
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, sep="\t", index=False)

    print(f"\u2705 Filtered gene variants saved to {output_path}")
    return df

# Example usage:
# extract_gene_variants("data/haploid_vcf_ann.vcf", output_path=analysis_results/variants_analysis/files/genes_of_interest.tsv")
